In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, global_mean_pool
from torch_geometric.data import Data
import networkx as nx
import numpy as np
import pandas as pd
import sys
sys.path.append('..')
from envs.routing_env import RoutingEnv

# Load trained model
class GNNDQNPolicy(nn.Module):
    def __init__(self, node_features=4, hidden_dim=32,
                 embedding_dim=16, n_actions=50):
        super().__init__()
        self.conv1 = GCNConv(node_features, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, embedding_dim)
        self.fc1 = nn.Linear(embedding_dim, 64)
        self.fc2 = nn.Linear(64, n_actions)

    def forward(self, x, edge_index, batch=None):
        h = F.relu(self.conv1(x, edge_index))
        h = F.relu(self.conv2(h, edge_index))
        if batch is None:
            batch = torch.zeros(x.shape[0], dtype=torch.long)
        graph_embed = global_mean_pool(h, batch)
        q = F.relu(self.fc1(graph_embed))
        return self.fc2(q)

policy = GNNDQNPolicy()
policy.load_state_dict(torch.load('../results/gnn_dqn_weights.pt',
                                   weights_only=False))
policy.eval()

# Build NEW 80-node unseen graph
np.random.seed(99)  # different seed — never seen during training
G_new = nx.erdos_renyi_graph(80, 0.12, seed=99)
while not nx.is_connected(G_new):
    G_new = nx.erdos_renyi_graph(80, 0.12, seed=np.random.randint(1000))

for u, v in G_new.edges():
    G_new[u][v]['latency'] = round(np.random.uniform(1, 5), 2)

deg_new = nx.degree_centrality(G_new)
bet_new = nx.betweenness_centrality(G_new)
clo_new = nx.closeness_centrality(G_new)
nodes_new = list(G_new.nodes())

# Convert to PyG
node_features = []
for node in G_new.nodes():
    node_features.append([deg_new[node], bet_new[node],
                          clo_new[node], np.random.uniform(0, 1)])
x_new = torch.tensor(node_features, dtype=torch.float)

edge_list = []
for u, v in G_new.edges():
    edge_list.append([u, v])
    edge_list.append([v, u])
edge_index_new = torch.tensor(edge_list, dtype=torch.long).t().contiguous()
data_new = Data(x=x_new, edge_index=edge_index_new)

print("New graph: nodes =", G_new.number_of_nodes(),
      "edges =", G_new.number_of_edges())

In [ ]:
# Test GNN-DQN on unseen 80-node graph
from envs.routing_env import RoutingEnv

env_new = RoutingEnv(n_nodes=80, edge_prob=0.12, seed=99)
results = []

for ep in range(100):
    obs, _ = env_new.reset()
    done = False
    total_reward = 0
    hops = 0
    while not done:
        with torch.no_grad():
            # Use new graph's PyG data
            q_vals = policy(data_new.x, data_new.edge_index)
            # Only consider valid actions for 80-node graph
            action = q_vals[0, :80].argmax().item()
        obs, reward, terminated, truncated, _ = env_new.step(action)
        done = terminated or truncated
        total_reward += reward
        hops += 1
    delivered = total_reward > 5
    results.append({'delivered': delivered, 'hops': hops,
                    'reward': total_reward})

gen_df = pd.DataFrame(results)
pdr  = round(gen_df['delivered'].mean() * 100, 2)
avgh = round(gen_df[gen_df['delivered']]['hops'].mean(), 2)
print(f"Generalisation (80-node unseen graph): PDR={pdr}%  Avg Hops={avgh}")
gen_df.to_csv('../results/generalisation_result.csv', index=False)
print("Saved generalisation_result.csv")